## Active ETF check

Load the **Timeseries** sheet of `CLEAN_RECOLTE DONNEES-EXCEL.xlsx` and keep the funds whose category is **Active ETF** (`fund_wrapper` ETF, `fund_type` Active).

Fund columns on that sheet are monthly total returns (`P(t) / P(t-1) - 1`), not price levels. The only price-level column is **S&P 500 Price**. Each ISIN is matched to a market identifier in `cache/cache-isin-openfigi.json` and shown as the second column header.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd


def _ensure_helpers_importable() -> None:
    """Put code/ on sys.path whether the kernel cwd is code/ or notebooks/."""
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            root = str(directory)
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


_ensure_helpers_importable()

from helpers.timeseries_functions.load_workbook import load_workbook

FILENAME = "CLEAN_RECOLTE DONNEES-EXCEL.xlsx"
RELATIVE = Path("input-data-alix") / "clean"
OPENFIGI_CACHE = Path("cache") / "cache-isin-openfigi.json"
META_LABELS = {"name", "category", "fee (%/year)", "Note", "Date"}


def find_cache(cache_relative: Path) -> Path:
    """Find a cache file whether the kernel cwd is code/ or notebooks/."""
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        candidate = directory / cache_relative
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {cache_relative}, starting from {start}")

timeseries_df = load_workbook(RELATIVE, FILENAME)

category_row = timeseries_df.loc[timeseries_df["id"].eq("category")].iloc[0]
name_row = timeseries_df.loc[timeseries_df["id"].eq("name")].iloc[0]
active_etf_isins = category_row.index[category_row.eq("Active ETF")].tolist()

active_etf_returns = timeseries_df.loc[
    ~timeseries_df["id"].isin(META_LABELS), ["id", *active_etf_isins]
].rename(columns={"id": "Date"})
active_etf_returns["Date"] = pd.to_datetime(active_etf_returns["Date"])
active_etf_returns = (
    active_etf_returns.set_index("Date")
    .apply(pd.to_numeric, errors="coerce")
    .sort_index()
)
isin_cache = json.loads(find_cache(OPENFIGI_CACHE).read_text())
tickers = {
    isin: ticker
    for isin, ticker in isin_cache.items()
    if isinstance(ticker, str) and ticker
}
market_identifiers = [tickers.get(isin) for isin in active_etf_returns.columns]
active_etf_returns.columns = pd.MultiIndex.from_arrays(
    [list(active_etf_returns.columns), market_identifiers],
    names=["ISIN", "market_identifier"],
)

index_return = pd.to_numeric(
    timeseries_df.loc[~timeseries_df["id"].isin(META_LABELS), "S&P 500 TR return"],
    errors="coerce",
)
index_return.index = pd.to_datetime(
    timeseries_df.loc[~timeseries_df["id"].isin(META_LABELS), "id"]
)
active_etf_returns.insert(
    0,
    ("S&P 500 TR", "Index"),
    index_return.reindex(active_etf_returns.index),
)

active_etf_names = name_row.reindex(active_etf_isins).astype("string")
active_etf_names.index.name = "ISIN"
active_etf_names.name = "Name"

mapped = sum(ticker is not None for ticker in market_identifiers)
print(f"Active ETF funds: {len(active_etf_isins)}")
print(f"Market identifiers: {mapped} of {len(market_identifiers)}")
print(
    "S&P 500 TR return: "
    f"{int(active_etf_returns[('S&P 500 TR', 'Index')].notna().sum())} months"
)
print(f"Months: {len(active_etf_returns)} ({active_etf_returns.index.min():%b %Y} – {active_etf_returns.index.max():%b %Y})")
print(f"Non-null monthly returns: {int(active_etf_returns.notna().sum().sum())}")

active_etf_returns

Active ETF funds: 68
Market identifiers: 68 of 68
S&P 500 TR return: 201 months
Months: 202 (Jan 2010 – Oct 2026)
Non-null monthly returns: 3598


ISIN,S&P 500 TR,US30151E4677,US00768Y7682,US00888H8777,US00888H8694,US00888H1095,US00888H7035,US00888H3075,US00888H6045,US00888H2085,...,US2103228225,US2103228308,US2103228639,US2103228480,US2103227987,US2103227649,US2103226815,US2103227722,US2103227805,US45259A1593
market_identifier,Index,RMRC,VEGA,SIXO,SIXJ,APRT,JANT,JULT,OCTT,APRW,...,JULZ,JUNZ,MARZ,MAYZ,SEPZ,DECZ,JANZ,NOVZ,OCTZ,VOOB
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-02-01,-0.022220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-03-01,0.026827,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-01,0.057478,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-05-01,0.021508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-01,0.052372,0.005933,-0.002412,0.003699,0.007103,0.000714,-0.001104,0.005518,0.000479,0.001270,...,-0.013622,-0.007243,-0.009736,-0.008523,-0.012027,-0.010394,-0.010531,-0.009923,-0.011320,NaN
2026-07-01,-0.014283,0.007519,-0.008495,0.006791,0.003363,0.006717,0.006251,0.002332,0.006806,0.005816,...,-0.000293,-0.000701,-0.001468,-0.001008,-0.002045,-0.001382,-0.001330,-0.001685,-0.001901,NaN


In [2]:
import yfinance as yf

# Daily Yahoo history, then a monthly total return on the same month labels as Lipper.
# Close is split-adjusted. Dividends and capital gains are reinvested on the ex-date,
# the same total-return level used in the fund timeseries checks.

pairs = [
    (isin, ticker)
    for isin, ticker in zip(
        active_etf_returns.columns.get_level_values("ISIN"),
        active_etf_returns.columns.get_level_values("market_identifier"),
        strict=True,
    )
    if isin != "S&P 500 TR"
]
start = active_etf_returns.index.min() - pd.Timedelta(days=40)
end = active_etf_returns.index.max() + pd.Timedelta(days=40)

downloaded = yf.download(
    [ticker for _, ticker in pairs],
    start=start.date().isoformat(),
    end=end.date().isoformat(),
    interval="1d",
    auto_adjust=False,
    actions=True,
    group_by="ticker",
    threads=True,
    progress=False,
)


def yahoo_total_return_level(daily: pd.DataFrame) -> pd.Series:
    daily = daily.dropna(subset=["Close"]).copy()
    daily.index = pd.to_datetime(daily.index).tz_localize(None)
    close = daily["Close"].astype(float)
    distribution = daily["Dividends"].fillna(0.0) if "Dividends" in daily.columns else 0.0
    if "Capital Gains" in daily.columns:
        distribution = distribution + daily["Capital Gains"].fillna(0.0)
    growth = (close + distribution) / close.shift(1)
    growth.iloc[0] = 1.0
    return close.iloc[0] * growth.cumprod()


levels: dict[tuple[str, str], pd.Series] = {}
for isin, ticker in pairs:
    if ticker not in downloaded.columns.get_level_values(0):
        continue
    history = downloaded[ticker]
    if history.empty or history["Close"].dropna().empty:
        continue
    levels[(isin, ticker)] = yahoo_total_return_level(history)

active_etf_yahoo_daily = pd.DataFrame(levels).sort_index()
active_etf_yahoo_daily.columns = pd.MultiIndex.from_tuples(
    active_etf_yahoo_daily.columns,
    names=["ISIN", "market_identifier"],
)
active_etf_yahoo_daily.index.name = "Date"

month_end_level = active_etf_yahoo_daily.resample("ME").last()
yahoo_monthly = month_end_level / month_end_level.shift(1) - 1
yahoo_monthly.index = yahoo_monthly.index.to_period("M").to_timestamp()

lipper = active_etf_returns.copy()
lipper.columns = pd.MultiIndex.from_arrays(
    [
        lipper.columns.get_level_values("ISIN"),
        lipper.columns.get_level_values("market_identifier"),
        ["Lipper"] * lipper.shape[1],
    ],
    names=["ISIN", "market_identifier", "source"],
)
yahoo_monthly.columns = pd.MultiIndex.from_arrays(
    [
        yahoo_monthly.columns.get_level_values("ISIN"),
        yahoo_monthly.columns.get_level_values("market_identifier"),
        ["Yahoo"] * yahoo_monthly.shape[1],
    ],
    names=["ISIN", "market_identifier", "source"],
)
ordered = [("S&P 500 TR", "Index", "Lipper")] + [
    (isin, ticker, source)
    for isin, ticker in pairs
    for source in ("Lipper", "Yahoo")
]
active_etf_returns = lipper.join(yahoo_monthly, how="left").reindex(
    columns=pd.MultiIndex.from_tuples(
        ordered, names=["ISIN", "market_identifier", "source"]
    )
)

found = active_etf_yahoo_daily.shape[1]
print(f"Yahoo daily total-return levels: {found} of {len(pairs)} funds, {len(active_etf_yahoo_daily)} sessions")
print(
    "Yahoo monthly returns added beside Lipper: "
    f"{int(active_etf_returns.xs('Yahoo', level='source', axis=1).notna().sum().sum())} months"
)

active_etf_returns

Yahoo daily total-return levels: 68 of 68 funds, 3545 sessions
Yahoo monthly returns added beside Lipper: 3441 months


ISIN              S&P 500 TR US30151E4677           US00768Y7682            \
market_identifier      Index         RMRC                   VEGA             
source                Lipper       Lipper     Yahoo       Lipper     Yahoo   
Date                                                                         
2010-01-01               NaN          NaN       NaN          NaN       NaN   
2010-02-01         -0.022220          NaN       NaN          NaN       NaN   
2010-03-01          0.026827          NaN       NaN          NaN       NaN   
2010-04-01          0.057478          NaN       NaN          NaN       NaN   
2010-05-01          0.021508          NaN       NaN          NaN       NaN   
...                      ...          ...       ...          ...       ...   
2026-06-01          0.052372     0.005933  0.008325    -0.002412 -0.001899   
2026-07-01         -0.014283     0.007519  0.006314    -0.008495 -0.009705   
2026-08-01          0.016359     0.013354  0.012549     0.020000  0.021330   
2026-09-01          0.005092    -0.035393 -0.037568    -0.014257 -0.014111   
2026-10-01          0.005570     0.032258  0.037022     0.001871 -0.006298   

ISIN              US00888H8777           US00888H8694           US00888H1095  \
market_identifier         SIXO                   SIXJ                   APRT   
source                  Lipper     Yahoo       Lipper     Yahoo       Lipper   
Date                                                                           
2010-01-01                 NaN       NaN          NaN       NaN          NaN   
2010-02-01                 NaN       NaN          NaN       NaN          NaN   
2010-03-01                 NaN       NaN          NaN       NaN          NaN   
2010-04-01                 NaN       NaN          NaN       NaN          NaN   
2010-05-01                 NaN       NaN          NaN       NaN          NaN   
...                        ...       ...          ...       ...          ...   
2026-06-01            0.003699  0.004031     0.007103  0.006345     0.000714   
2026-07-01            0.006791  0.007499     0.003363  0.004386     0.006717   
2026-08-01            0.008775  0.008080     0.016813  0.017467     0.015273   
2026-09-01            0.004474  0.005462     0.000924  0.000536     0.002996   
2026-10-01            0.003550 -0.002184     0.004431 -0.001180     0.004395   

ISIN               ... US2103227649           US2103226815            \
market_identifier  ...         DECZ                   JANZ             
source             ...       Lipper     Yahoo       Lipper     Yahoo   
Date               ...                                                 
2010-01-01         ...          NaN       NaN          NaN       NaN   
2010-02-01         ...          NaN       NaN          NaN       NaN   
2010-03-01         ...          NaN       NaN          NaN       NaN   
2010-04-01         ...          NaN       NaN          NaN       NaN   
2010-05-01         ...          NaN       NaN          NaN       NaN   
...                ...          ...       ...          ...       ...   
2026-06-01         ...    -0.010394 -0.009052    -0.010531 -0.009768   
2026-07-01         ...    -0.001382 -0.000902    -0.001330 -0.001221   
2026-08-01         ...     0.020625  0.019744     0.020796  0.019997   
2026-09-01         ...    -0.004748 -0.004108    -0.004379 -0.004673   
2026-10-01         ...     0.006316 -0.005539     0.006146 -0.004744   

ISIN              US2103227722           US2103227805           US45259A1593  \
market_identifier         NOVZ                   OCTZ                   VOOB   
source                  Lipper     Yahoo       Lipper     Yahoo       Lipper   
Date                                                                           
2010-01-01                 NaN       NaN          NaN       NaN          NaN   
2010-02-01                 NaN       NaN          NaN       NaN          NaN   
2010-03-01                 NaN       NaN          NaN       NaN    

In [ ]:
# Lipper has no price level on this sheet, only a monthly return.
# Yahoo beside it is the month-end total return (dividends and capital gains reinvested).
# A gap of a few basis points is the usual NAV-date difference. A large gap is checked
# against the sheet's S&P 500 TR return.

CLOSE = 1e-3  # 10 bp
WIDE = 5e-3  # 50 bp

lipper = active_etf_returns.xs("Lipper", level="source", axis=1)
index_return = lipper.pop(("S&P 500 TR", "Index"))
yahoo = active_etf_returns.xs("Yahoo", level="source", axis=1)
common = lipper.columns.intersection(yahoo.columns)
lipper = lipper[common]
yahoo = yahoo[common]

both = lipper.notna() & yahoo.notna()
gap = (lipper - yahoo).where(both)
abs_gap = gap.abs()

fund_rows = []
for column in common:
    isin, ticker = column
    paired = both[column]
    n = int(paired.sum())
    if n < 6:
        continue
    distance = abs_gap.loc[paired, column]
    lipper_vs_index = lipper.loc[paired, column].corr(index_return.loc[paired])
    yahoo_vs_index = yahoo.loc[paired, column].corr(index_return.loc[paired])
    fund_rows.append(
        {
            "Ticker": ticker,
            "Name": active_etf_names.get(isin, ""),
            "Months": n,
            "Median gap (bp)": distance.median() * 1e4,
            "Within 10 bp": (distance <= CLOSE).mean(),
            "Over 50 bp": (distance > WIDE).mean(),
            "Correlation": lipper.loc[paired, column].corr(yahoo.loc[paired, column]),
            "Lipper vs S&P": lipper_vs_index,
            "Yahoo vs S&P": yahoo_vs_index,
        }
    )

comparison = pd.DataFrame(fund_rows).sort_values("Median gap (bp)", ascending=False)
flat = abs_gap.to_numpy()[both.to_numpy()]
signed = gap.to_numpy()[both.to_numpy()]
wide = both & (abs_gap > WIDE)
lipper_to_index = lipper.sub(index_return, axis=0).abs()
yahoo_to_index = yahoo.sub(index_return, axis=0).abs()

# The same Yahoo levels, measured to the sheet date (the 1st) instead of month-end.
on_sheet_date = active_etf_yahoo_daily.reindex(columns=common).reindex(
    lipper.index, method="ffill"
)
yahoo_on_sheet_date = on_sheet_date / on_sheet_date.shift(1) - 1
sheet_corrs = []
for column in common:
    paired = lipper[column].notna() & yahoo_on_sheet_date[column].notna() & index_return.notna()
    if int(paired.sum()) < 6:
        continue
    sheet_corrs.append(yahoo_on_sheet_date.loc[paired, column].corr(index_return.loc[paired]))

print("Lipper against Yahoo month-end total return")
print(f"- Overlapping fund-months: {len(flat)}")
print(f"- Lipper only: {int((lipper.notna() & yahoo.isna()).to_numpy().sum())}")
print(f"- Yahoo only: {int((yahoo.notna() & lipper.isna()).to_numpy().sum())}")
print(f"- Median absolute gap: {pd.Series(flat).median() * 1e4:.1f} bp")
print(f"- Mean signed gap (Lipper minus Yahoo): {pd.Series(signed).mean() * 1e4:.1f} bp")
print(f"- Within 10 bp: {(flat <= CLOSE).mean():.1%}")
print(f"- Within 50 bp: {(flat <= WIDE).mean():.1%}")
print(
    f"- Fund correlation: median {comparison['Correlation'].median():.3f}, "
    f"{int((comparison['Correlation'] > 0.99).sum())} of {len(comparison)} above 0.99, "
    f"{int((comparison['Correlation'] < 0.80).sum())} below 0.80"
)
print(
    f"- Months wider than 50 bp: {int(wide.to_numpy().sum())}. "
    f"Yahoo is closer to the S&P 500 TR return in "
    f"{int((wide & (yahoo_to_index < lipper_to_index)).to_numpy().sum())}, "
    f"Lipper in {int((wide & (lipper_to_index < yahoo_to_index)).to_numpy().sum())}."
)
print(
    "- Median correlation of Yahoo with the sheet's S&P 500 TR return "
    f"on the sheet dates: {pd.Series(sheet_corrs).median():.3f}. "
    f"On the month-end dates used by Lipper it is {comparison['Yahoo vs S&P'].median():.3f}, "
    f"the same as Lipper ({comparison['Lipper vs S&P'].median():.3f})."
)

review = comparison.loc[comparison["Median gap (bp)"] > 20, [
    "Ticker", "Name", "Months", "Median gap (bp)", "Correlation",
]]
print("\nFunds whose median gap is wider than 20 bp")
if review.empty:
    print("- none")
else:
    print(review.round(3).to_string(index=False))

comparison.round(3)
